In [80]:
import pandas as pd

In [81]:
import warnings
warnings.filterwarnings('ignore')

In [82]:
data = pd.read_csv("Retail_Cleaned.csv")

In [83]:
## Q1. Dataset Audit
data.shape
data.dtypes
data.isna().sum()
data.duplicated().sum()
data.nunique()


Row ID            51290
Order ID          25035
Order Date         1430
Ship Date          1464
Order_year            4
Order_month          12
Quarter               4
Ship Mode             4
Shipping Cost     10037
Order Priority        4
Customer ID        1590
Customer Name       795
Segment               3
Market                7
Region               13
Country             147
City               3636
State              1094
Postal Code         631
Product ID        10292
Category              3
Sub-Category         17
Product Name       3788
Sales             22995
Quantity             14
Discount             23
Profit            24575
Profit Margin       263
Sales_Category        3
Profit_status         2
dtype: int64

In [84]:
##Q2. Data Cleaning
data['Ship Date'] = pd.to_datetime(data['Ship Date'] , errors='coerce')
data['Order Date'] = pd.to_datetime(data['Order Date'], errors='coerce')
data.isna().sum()
data['Postal Code'].isna().mean() * 100
data[data['Postal Code'].isna()]['Country'].value_counts()
## Postal Code → Missing values retained due to very high missingness (80.5%).
data.duplicated().sum()
data[data.duplicated()]
data['Category'].unique()
data['Segment'].unique()
data['Region'].unique()
data['Sub-Category'].unique()

array(['Accessories', 'Chairs', 'Phones', 'Copiers', 'Tables', 'Binders',
       'Supplies', 'Appliances', 'Machines', 'Bookcases', 'Storage',
       'Furnishings', 'Art', 'Paper', 'Envelopes', 'Fasteners', 'Labels'],
      dtype=object)

In [85]:
## Q3. Data Quality Validation 🔥
(data['Sales'] < 0).sum()
(data['Quantity']<=0).sum()
data['Discount'] = pd.to_numeric(data['Discount'], errors='coerce')
((data['Discount']<0) | (data['Discount']>1)).sum()
data['Profit'].describe()
data['Order Date'].isna().sum()
data['Ship Date'].isna().sum()

np.int64(0)

In [86]:
## Q4 — Overall Business KPIs 🔥
data['Sales'].sum()
data['Profit'].sum()
data['Quantity'].sum()
data['Order ID'].nunique()
data['Customer ID'].nunique()
data['Product ID'].nunique()
data['Sales'].sum()/data['Order ID'].nunique()
data['Discount'].mean()*100
data['Profit'].sum()/data['Sales'].sum()*100




np.float64(11.609233780477135)

In [87]:
## Q5. Category & Sub-Category Analysis 🔥
data.groupby(['Category','Sub-Category']).agg(
    total_sales = ('Sales', 'sum'),
    total_profit = ('Profit', 'sum'),
    total_quantity = ('Quantity', 'sum'),
    average_discount = ('Discount', 'mean')

)

data.groupby('Category')['Sales'].sum().idxmax()
data.groupby('Category')['Profit'].sum().idxmax()
data.groupby('Category')['Profit'].sum().idxmin()
data.groupby('Sub-Category')['Profit'].sum().loc[lambda x: x<0].index

Index(['Tables'], dtype='object', name='Sub-Category')

In [88]:
## Q6. Top & Bottom Products Analysis 🔥
data.groupby('Product ID').agg(
      total_sales = ('Sales', 'sum'),
    total_profit = ('Profit', 'sum'),
    total_quantity = ('Quantity', 'sum')
)

data.groupby('Product ID')['Sales'].sum().sort_values(ascending=False).head(10)
data.groupby('Product ID')['Profit'].sum().sort_values(ascending=False).head(10)
data.groupby('Product ID')['Profit'].sum().sort_values(ascending=True).head(10)
data.groupby('Product ID')['Quantity'].sum().sort_values(ascending=False).head(10)

Product ID
OFF-AR-10003651    163
OFF-BI-10002799    130
OFF-AR-10003829    117
OFF-BI-10001808    112
OFF-BI-10003708    111
FUR-CH-10003354    106
OFF-BI-10002570    102
OFF-AR-10000110     97
OFF-BI-10004195     92
OFF-ST-10004377     89
Name: Quantity, dtype: int64

In [89]:
## Q7. Region & State Performance 🔥
data.groupby('Region')['Sales'].sum().sort_values(ascending = False).head(10)
data.groupby('Region')['Profit'].sum().sort_values(ascending = False).head(10)
data.groupby('Region')['Profit'].sum().sort_values(ascending = True).head(10)
data.groupby('State')['Sales'].sum().sort_values(ascending = False).head(10)
data.groupby('State')['Profit'].sum().sort_values(ascending = False).head(10)
data.groupby('State')['Profit'].sum().loc[lambda x: x<0].index

Index([''Ajman', 'Abia', 'Abuja Capital Territory', 'Aceh', 'Adamawa', 'Adana',
       'Adiyaman', 'Afyonkarahisar', 'Aksaray', 'Akwa Ibom',
       ...
       'Yangon', 'Yaracuy', 'Yobe', 'Yoro', 'Zamfara', 'Zealand', 'Zeeland',
       'Zhambyl', 'Zulia', 'Šiauliai'],
      dtype='object', name='State', length=290)

In [90]:
data['State'].nunique()

1094

In [91]:
## Q8. Customer Performance 🔥
data.groupby('Customer ID')['Sales'].sum().sort_values(ascending = False).head(10)
data.groupby('Customer ID')['Profit'].sum().sort_values(ascending = False).head(10)
data.groupby('Customer ID')['Profit'].sum().sort_values(ascending = True).head(10)
data.groupby('Customer ID')['Quantity'].sum().sort_values(ascending = False).head(10)
data.groupby(['Customer ID','Customer Name'])['Profit'].sum().loc[lambda x: x<0].index


MultiIndex([(  'AA-645',        'Anna Andreadi'),
            ('AB-10255', 'Alejandro Ballentine'),
            (  'AB-105',        'Adrian Barton'),
            ('AC-10660',           'Anna Chung'),
            (  'AC-450',              'Amy Cox'),
            (  'AC-615',            'Ann Chong'),
            (  'AD-180',       'Alan Dominguez'),
            (  'AG-300',  'Aleksandra Gannaway'),
            (  'AG-390',       'Allen Goldenen'),
            (  'AG-495',      'Andrew Gjertsen'),
            ...
            ('TW-11025',    'Tamara Willingham'),
            ('TZ-11580',            'Tracy Zic'),
            ('VD-11670',    'Valerie Dominguez'),
            ('VG-21790',       'Vivek Gonzalez'),
            ('VM-11685',      'Valerie Mitchum'),
            ('VM-21685',      'Valerie Mitchum'),
            ('VP-11730',         'Victor Preis'),
            ('WB-11850',        'William Brown'),
            ('ZC-11910',     'Zuschuss Carroll'),
            ('ZD-21925',   'Zuschu

In [92]:
## Q9. Sales & Profit Trend Analysis 🔥
data.groupby('Order_year')[['Sales','Profit']].sum()
data.groupby('Order_month')[['Sales','Profit']].sum()
data.groupby(['Order_year','Order_month'])[['Sales','Profit']].sum()
data.groupby('Order_month')['Sales'].sum().idxmax()
data.groupby('Order_month')['Profit'].sum().idxmax()
data.groupby('Order_month')['Profit'].sum().idxmin()


'February'

In [93]:
## Q10. Discount & Profitability Analysis 🔥


data['Discount'] = (
    data['Discount']
    .astype(str)
    .str.replace('%', '', regex=False)
    .astype(float) / 100
)

data['Discount_Range'] = pd.cut(
    data['Discount'],
    bins=[0, 0.10, 0.20, 0.30, 1],
    labels=['0-10%', '10-20%', '20-30%', '30%+'],
    include_lowest=True
)
data.groupby('Discount_Range')['Sales'].sum()
data.groupby('Discount_Range')['Profit'].sum()
data.groupby('Discount_Range').apply(
    lambda x:(x['Profit']/x['Sales']).mean()
)
data.groupby('Discount_Range')['Profit'].sum().idxmax()
data.groupby('Discount_Range')['Profit'].sum().loc[lambda x: x<0].index
data[data['Discount'] >= 0.30]['Profit'].sum()

np.float64(0.0)

In [94]:
## Q11. Customer Segment & Profitability Analysis 🔥


data['Profit Margin'] = (
    data['Profit Margin']
    .astype(str)
    .str.replace('%', '', regex=False)
    .astype(float) / 100
)
data.groupby('Segment')['Sales'].sum()
data.groupby('Segment')['Profit'].sum()
data.groupby('Segment')['Quantity'].sum()
data.groupby('Segment')['Profit Margin'].mean()
data.groupby('Segment')['Sales'].sum().idxmax()




'Consumer'

In [95]:
## 13 Transform  


customer_data = data.groupby(
    ['Segment', 'Customer ID']
)['Sales'].sum().reset_index(name='Total_sales')

customer_data['Segment_avg'] = (
    customer_data.groupby('Segment')['Total_sales']
    .transform('mean')
)

customer_data['Difference'] = (
    customer_data['Total_sales'] -
    customer_data['Segment_avg']
)
customer_data

,Segment,Customer ID,Total_sales,Segment_avg,Difference
0,Consumer,AA-10315,13747.41300,7953.397821,5794.015179
1,Consumer,AA-10375,5884.19500,7953.397821,-2069.202821
2,Consumer,AA-10480,17695.58978,7953.397821,9742.191959
3,Consumer,AA-10645,15343.89070,7953.397821,7390.492879
4,Consumer,AA-315,2243.25600,7953.397821,-5710.141821
...,...,...,...,...,...
1585,Home Office,VM-21685,13708.12232,7803.564090,5904.558230
1586,Home Office,VP-11730,432.00000,7803.564090,-7371.564090
1587,Home Office,VP-21730,11629.26900,7803.564090,3825.704910
1588,Home Office,VT-11700,1271.44200,7803.564090,-6532.122090


In [96]:
## Q14 → Ranking 🔥
data.groupby('Customer ID')['Sales'].sum()
Customer_sales = data.groupby(['Segment','Customer ID'])['Sales'].sum().reset_index(name = 'Total_sales')
Customer_sales['rank'] = Customer_sales.groupby('Segment')['Total_sales'].rank(method = 'dense' , ascending = False)
Customer_sales
Customer_sales[Customer_sales['rank'] == 1]
data.groupby('Customer Name')['Sales'].sum().head(10)


Customer Name
Aaron Bergman         24644.62750
Aaron Hawkins         20759.51384
Aaron Smayling        14212.62840
Adam Bellavance       20186.77840
Adam Hart             21718.20142
Adam Shillingsburg    15444.67672
Adrian Barton         25123.18000
Adrian Hane           11405.91788
Adrian Shami          11286.05420
Aimee Bixby           16201.17460
Name: Sales, dtype: float64

In [97]:
## Q15 → Outliers 🔥
Q1 = data['Sales'].quantile(0.25)
Q2 = data['Sales'].quantile(0.75)

IQR = Q2 -Q1

Lower_sales = Q1-1.5*IQR
higher_sales = Q2+1.5*IQR

sales_outliers = data[(data['Sales']<Lower_sales) | (data['Sales']>higher_sales)]

Lower_sales
higher_sales
sales_outliers

,Row ID,Order ID,Order Date,Ship Date,Order_year,Order_month,Quarter,Ship Mode,Shipping Cost,Order Priority,...,Sub-Category,Product Name,Sales,Quantity,Discount,Profit,Profit Margin,Sales_Category,Profit_status,Discount_Range
1,26341,IN-2013-77878,2013-02-05,2013-02-07,2013,February,1,Second Class,923.63,Critical,...,Chairs,"Novimex Executive Leather Armchair, Black",3709.395,9,NaN,-288.7650,-0.08,High,Loss,NaN
2,25330,IN-2013-71249,2013-10-17,2013-10-18,2013,October,4,First Class,915.49,Medium,...,Phones,"Nokia Smart Phone, with Caller ID",5175.171,9,NaN,919.9710,0.18,High,Profit,NaN
3,13524,ES-2013-1579342,2013-01-28,2013-01-30,2013,January,1,First Class,910.16,Medium,...,Phones,"Motorola Smart Phone, Cordless",2892.510,5,NaN,-96.5400,-0.03,High,Loss,NaN
4,47221,SG-2013-4320,2013-11-05,2013-11-06,2013,November,4,Same Day,903.04,Critical,...,Copiers,"Sharp Wireless Fax, High-Speed",2832.960,8,NaN,311.5200,0.11,High,Profit,NaN
5,22732,IN-2013-42360,2013-06-28,2013-07-01,2013,June,2,Second Class,897.35,Critical,...,Phones,"Samsung Smart Phone, with Caller ID",2862.675,5,NaN,763.2750,0.27,High,Profit,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44220,23566,IN-2012-40029,2012-01-27,2012-02-02,2012,January,1,Standard Class,1.37,Medium,...,Chairs,"Office Star Swivel Stool, Set of Two",1585.710,9,NaN,0.0000,0.00,High,Loss,NaN
45234,10633,ES-2014-3066003,2014-09-15,2014-09-21,2014,September,3,Standard Class,1.19,Low,...,Bookcases,"Ikea Floating Shelf Set, Pine",618.084,4,NaN,27.4440,0.04,High,Profit,NaN
45351,12641,ES-2014-3582654,2014-10-23,2014-10-26,2014,October,4,First Class,1.17,Medium,...,Machines,"Panasonic Inkjet, Wireless",936.270,3,NaN,65.5200,0.07,High,Profit,NaN
47131,268,MX-2014-159282,2014-07-21,2014-07-26,2014,July,3,Standard Class,0.88,Medium,...,Chairs,"Novimex Executive Leather Armchair, Adjustable",607.360,2,NaN,133.6000,0.22,High,Profit,NaN
